In [6]:
import os
import tiktoken
import requests
import pandas as pd
import numpy as np
from tqdm import tqdm
from datetime import datetime
from openai import AzureOpenAI
from dotenv import dotenv_values
from azure.ai.ml import MLClient
from azure.identity import EnvironmentCredential, ManagedIdentityCredential, get_bearer_token_provider
from openai import AzureOpenAI
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    SearchFieldDataType,
    SimpleField,
    SearchableField,
    SearchIndex,
    SearchField,
    VectorSearch,
    VectorSearchProfile,
    HnswAlgorithmConfiguration,
)
from gettext import translation
from requests.exceptions import HTTPError
from azure.ai.translation.text import TextTranslationClient
from azure.ai.translation.text.models import InputTextItem
from azure.core.exceptions import HttpResponseError, ResourceNotFoundError, AzureError
from io import BytesIO
from azure.storage.blob import BlobServiceClient

In [7]:
env_config_file = '.env'
index_name = "measures-data"
service_endpoint = "https://azsearch-tutorial-001.search.windows.net/"
azure_endpoint = "https://openai-tutorial-001.openai.azure.com/"
embedding_model = "text-embedding-ada-002"
api_version = "2023-05-15"
translations_container_name='translated-measures'

In [8]:
if env_config_file != 'None':
    if os.path.isfile(env_config_file):
        config = dotenv_values(env_config_file)
        os.environ["AZURE_TENANT_ID"] = config["AZURE_TENANT_ID"]
        os.environ["AZURE_CLIENT_ID"] = config["AZURE_CLIENT_ID"]
        os.environ["AZURE_CLIENT_SECRET"] = config["AZURE_CLIENT_SECRET"]
        credential = EnvironmentCredential()

    print(credential)

# Initialize Search & Search Index Client
index_client = SearchIndexClient(endpoint=service_endpoint, index_name=index_name, credential=credential)
search_client = index_client.get_search_client(index_name)

In [10]:
def retrieve_measures_from_AI_search_service() -> pd.DataFrame:
    """Retrieves the state of measures from an AI Search Service.

       The function checks if the specified index exists and retrieves data from the service. 
       If the index is empty, it returns an empty DataFrame with predefined columns.

    Global Variables:
        service_endpoint (str): The endpoint for the AI Search Service.
        index_name (str): The name of the index to query.

    Raises:
        Exception: If the specified index does not exist in the AI Search Service.

    Returns:
        pd.DataFrame: A DataFrame containing the retrieved measures with the following columns:
            - "MeasureId"
            - "LastUpdatedAt"
    """

    # Print status message
    print("----------------------------------------------------------------------------------------------------")
    print("Step 3: Retrieve data from AI Search Service to get state of measures...")

    try:
        # Get the index configuration
        index = index_client.get_index(name=index_name)

        if index:
            dimensions_set = set()  # Initialize the set for dimensions
            for field in index.fields:
                if hasattr(field, 'vector_search_dimensions') and field.vector_search_dimensions:
                    print(f"Field: {field.name}, Dimensions: {field.vector_search_dimensions}")
                    dimensions_set.add(field.vector_search_dimensions)

            # Check if all dimensions are the same
            if len(dimensions_set) == 1:
                print("All fields have the same dimensions:", dimensions_set)
                embedding_vector_size = int(dimensions_set.pop())  # Safe to pop as there's only one element
                print(embedding_vector_size)
            elif len(dimensions_set) == 0:
                print("No fields with 'vector_search_dimensions' found.")
                embedding_vector_size = None  # Handle no dimensions found
            else:
                print("Fields have different dimensions:", dimensions_set)
                embedding_vector_size = None  # Handle varying dimensions 
    except Exception as e:
        print(f"No index found with the name '{index_name}'. Attempting to create the index...")
        embedding_vector_size = create_search_index(index_name, service_endpoint)
        
    # Check if index is empty
    try:
        results = search_client.search(search_text="*", top=1)
        is_empty = not any(results)  # any(results) is False when index exists but no content
    except Exception as e:
        print(f"Error checking if index is empty: {e}")

    if is_empty == True:
        # create an empty dataframe with the schemas as that in azure search and then initailize the embeddings
        
        # Define logic to do a full load where embeddings are pre-calculated
        pass
    else:
        # Should point to a different folder in blob and only do a merge, update and delete but not read from vector
        # where embeddings are pre-calculated
        pass
    return embedding_vector_size


def create_search_index(index_name: str, service_endpoint: str):
    """Create a search index with the specified index name and service endpoint.

    Args:
        index_name (str): The name of the search index.
        service_endpoint (str): The service endpoint for the search index.

    Returns:
        int: The size of the embedding vector used for vector search, or None if an error occurs.
    """
    global embedding_model

    try:
        # Get embedding model
        embedding_model = embedding_model

        vector_embedding = calculate_embeddings("Test for embedding size", embedding_model)
        
        # Identify the vector size
        embedding_vector_size = len(vector_embedding)

        # Define the index fields
        fields = [
            SimpleField(
                name="MeasureId",
                type=SearchFieldDataType.String,
                key=True,
                filterable=True,
                sortable=True
            ),
            SimpleField(
                name="LastUpdatedAt",
                type=SearchFieldDataType.String,
                filterable=True,
                sortable=True
            ),
            SimpleField(
                name="Deleted",
                type=SearchFieldDataType.Int32,
            ),
            SearchableField(
                name="Name", 
                type=SearchFieldDataType.String,
            ),
            SearchField(
                name="Name_vector",
                type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
                vector_search_dimensions=embedding_vector_size,
                vector_search_profile_name="my-vector-config",
            ),
            SearchableField(
                name="Description", 
                type=SearchFieldDataType.String,
            ),
            SearchField(
                name="Description_vector",
                type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
                vector_search_dimensions=embedding_vector_size,
                vector_search_profile_name="my-vector-config",
            ),
            SearchableField(
                name="BusinessRequirement", 
                type=SearchFieldDataType.String,
            ),
            SearchField(
                name="BusinessRequirement_vector",
                type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
                vector_search_dimensions=embedding_vector_size,
                vector_search_profile_name="my-vector-config",
            ),
            SearchableField(
                name="InitialState", 
                type=SearchFieldDataType.String,
            ),
            SearchField(
                name="InitialState_vector",
                type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
                vector_search_dimensions=embedding_vector_size,
                vector_search_profile_name="my-vector-config",
            ),
            SearchableField(
                name="TargetState", 
                type=SearchFieldDataType.String,
            ),
            SearchField(
                name="TargetState_vector",
                type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
                vector_search_dimensions=embedding_vector_size,
                vector_search_profile_name="my-vector-config",
            ),
            SearchableField(
                name="Analysis", 
                type=SearchFieldDataType.String,
            ),
            SearchField(
                name="Analysis_vector",
                type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
                vector_search_dimensions=embedding_vector_size,
                vector_search_profile_name="my-vector-config",
            ),
            SearchableField(
                name="Remarks", 
                type=SearchFieldDataType.String,
            ),
            SearchField(
                name="Remarks_vector",
                type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
                vector_search_dimensions=embedding_vector_size,
                vector_search_profile_name="my-vector-config",
            ),
            SearchableField(
                name="Results", 
                type=SearchFieldDataType.String,
            ),
            SearchField(
                name="Results_vector",
                type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
                vector_search_dimensions=embedding_vector_size,
                vector_search_profile_name="my-vector-config",
            ),
            SearchableField(
                name="LessonsLearned", 
                type=SearchFieldDataType.String,
            ),
            SearchField(
                name="LessonsLearned_vector",
                type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
                vector_search_dimensions=embedding_vector_size,
                vector_search_profile_name="my-vector-config",
            ),
        ]
        vector_search = VectorSearch(  
            profiles=[VectorSearchProfile(name="my-vector-config", algorithm_configuration_name="my-algorithms-config")],  
            algorithms=[HnswAlgorithmConfiguration(name="my-algorithms-config")],  
        )  

        # Create the search index client
        search_client = SearchIndexClient(service_endpoint, credential)

        # Create the search index
        search_index = SearchIndex(name=index_name, fields=fields, vector_search=vector_search)
        result = search_client.create_index(search_index)
        
        # Print status message
        print(f"Index {result.name} created")

        return embedding_vector_size
    except Exception as e:
        print(f"Error creating search index: {e}")
        return None


def calculate_embeddings(text: str, embedding_model: str = "text-embedding-ada-002") -> list:
    
    """Calculates the embedding vector for a given text using the specified Azure OpenAI embedding model.

    Args:
        azure_client (AzureOpenAI): An instance of the Azure OpenAI client, configured for the embedding service.
        text (str): The input text for which the embedding needs to be generated.
        embedding_model (str, optional): The name of the embedding model to use. Defaults to "text-embedding-ada-002".

    Returns:
        list: A list of floats representing the embedding vector for the input text, or None if an error occurs.
    """
    try:
        # Fetch Azure AD token using Managed Identity
        token = credential.get_token("https://cognitiveservices.azure.com/.default").token

        # Make request using Azure AD token
        headers = {
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json"
        }

        url = f"{azure_endpoint}openai/deployments/{embedding_model}/embeddings?api-version={api_version}"
        payload = {"input": text}

        response = requests.post(url, headers=headers, json=payload)
        response_data = response.json()

        # Extract the embedding
        if response.status_code == 200:
            embedding = response_data['data'][0]['embedding']
            return embedding
        else:
            print(f"Error: {response.status_code}, {response.text}")
            return None
    except Exception as e:
        print(f"Exception occurred while calculating embeddings: {e}")
        return None

In [14]:
retrieve_measures_from_AI_search_service()

----------------------------------------------------------------------------------------------------
Step 3: Retrieve data from AI Search Service to get state of measures...
No index found with the name 'measures-data'. Attempting to create the index...
Index measures-data created
Index exists


In [24]:
index_client.delete_index('measures-data')
print("Index deleted")

Index deleted


In [9]:
file_name = 'translation_test.csv'
container_name = 'translated-measures'

# Initialize the Blob Service Client
blob_service_client = BlobServiceClient(
    account_url="https://satutorial001.blob.core.windows.net/", credential=credential)
container_client = blob_service_client.get_container_client(container=container_name)
blob_list = container_client.list_blobs()

try:
    for blob in blob_list:
        print(blob.name)
        if blob.name == file_name:
            blob_client = container_client.get_blob_client(blob)
            blob_data = blob_client.download_blob().readall()
            df_existing = pd.read_csv(BytesIO(blob_data))
            # Convert the DataFrame to a Parquet file
            parquet_buffer = BytesIO()
            df_existing.to_parquet(parquet_buffer, index=False)
            parquet_buffer.seek(0)

            # Upload the Parquet file to the same container
            parquet_blob_client = container_client.get_blob_client("Measures.parquet")
            parquet_blob_client.upload_blob(parquet_buffer, overwrite=True)
            print("File uploaded as Measures.parquet")
            break
    else:
        print(f"Blob '{file_name}' not found in container '{container_name}'.")
except Exception as e:
    print(f"An error occurred: {e}")

translation_test.csv
File uploaded as Measures.parquet


In [11]:
def read_translated_measures_data_for_initial_load():
    file_name = "Measures.parquet"  
    # Initialize the Blob Service Client
    blob_service_client = BlobServiceClient(
        account_url="https://satutorial001.blob.core.windows.net", credential=credential)
    container_client = blob_service_client.get_container_client(
        container='translated-measures')
    blob_client = container_client.get_blob_client(blob=file_name)

    if blob_client.exists():
        blob_data = blob_client.download_blob()
        df_translations = pd.read_parquet(BytesIO(blob_data.readall()))

    df_translations['lastUpdatedAt'] = pd.to_datetime(
        df_translations['lastUpdatedAt'], errors='coerce')
    df_translations['createdAt'] = pd.to_datetime(
        df_translations['createdAt'])
    df_translations['endDate'] = pd.to_datetime(
        df_translations['endDate'], errors='coerce')
    df_translations['startDate'] = pd.to_datetime(
        df_translations['startDate'], errors='coerce')
    df_translations[['analysis','results','lessonsLearned']] = df_translations[['analysis','results','lessonsLearned']].astype(str)
    df_translations['deleted'] = 0
    

    return df_translations


In [12]:
def reduce_raw_dataframe_to_relevant_columns(df: pd.DataFrame, columns_relevant: list) -> pd.DataFrame:
    """Reduces the input DataFrame to only the specified relevant columns and adjusts the column names
       to start with an uppercase letter.

    Args:
        df (pd.DataFrame): The input raw DataFrame to be processed.
        columns_relevant (list): A list of column names to retain in the DataFrame.

    Returns:
        pd.DataFrame: A DataFrame containing only the relevant columns, with column names starting 
                      with an uppercase letter, or None if an error occurs.
    """
    try:
        # Print status message
        print("----------------------------------------------------------------------------------------------------")
        print("Step 2: Reduce measures to relevant columns...")

        # Reduce to relevant columns
        df = df[columns_relevant]

        # Switch to upper case
        df.columns = [column[0].upper() + column[1:] for column in df.columns]

        # Print status message
        print("Relevant measures:")
        print(df.head())

        return df
    except Exception as e:
        print(f"Error reducing DataFrame: {e}")
        return None

In [13]:
def initialize_embedding_vectors(df_measures_to_upload: pd.DataFrame, columns_relevant: list, embedding_vector_size: int) -> pd.DataFrame:
    """Initializes embedding vectors for relevant columns in the measures DataFrame.

    Args:
        df_measures_to_upload (pd.DataFrame): A DataFrame containing measures to upload, 
                                              including columns like "MeasureId", "LastUpdatedAt", and others requiring embeddings.
        columns_relevant (list): A list of column names for which embedding vectors should be initialized.
        embedding_vector_size (int): The size of the embedding vector used for vector search.

    Returns:
        pd.DataFrame: A DataFrame with initialized embedding vector columns and organized structure, or None if an error occurs.
    """
    try:
        # Print status message
        print("----------------------------------------------------------------------------------------------------")
        print("Step 6: Initialize embedding vectors...")
        # Upper case for columns_relevant
        columns_relevant = [column[0].upper() + column[1:] for column in columns_relevant]

        # Initialize a zero vector using NumPy for efficiency
        zero_vector = np.zeros(embedding_vector_size).tolist()

        # To avoid SettingWithCopyWarning
        df_reduced = df_measures_to_upload[columns_relevant].copy()

        # Create new data frame with columns for embeddings
        for column in tqdm(columns_relevant, total=len(columns_relevant), desc="Initialize vector embeddings"):
            if column not in ["MeasureId", "LastUpdatedAt", "Deleted"]:
                df_reduced[column + "_vector"] = [zero_vector.copy() for _ in range(len(df_reduced))]

        # Organize the columns
        df_reduced = df_reduced[
            [
                "MeasureId",
                "LastUpdatedAt",
                "Deleted",
                "Name",
                "Name_vector",
                "Description",
                "Description_vector",
                "BusinessRequirement",
                "BusinessRequirement_vector",
                "InitialState",
                "InitialState_vector",
                "TargetState",
                "TargetState_vector",
                "Analysis",
                "Analysis_vector",
                "Remarks",
                "Remarks_vector",
                "Results",
                "Results_vector",
                "LessonsLearned",
                "LessonsLearned_vector",
            ]
        ]

        # Convert MeasureId to string
        df_reduced["MeasureId"] = df_reduced["MeasureId"].astype(str)

        # Print status message
        print("Size of data frame with measures to update: {}".format(df_reduced.shape))
        print(df_reduced.head())

        return df_reduced
    except Exception as e:
        print(f"Error initializing embedding vectors: {e}")
        return None

In [14]:
def check_token_size_before_embedding(text: str, max_token_for_model: int, embedding_model: str = "text-embedding-ada-002") -> bool:
    """Checks if the number of tokens in a given text is within the allowable limit for a specific embedding model.

    Args:
        text (str): The input text to be evaluated.
        max_token_for_model (int): The maximum number of tokens allowed for the specified embedding model.
        embedding_model (str, optional): The name of the embedding model to use for token encoding.
                                         Defaults to "text-embedding-3-small".

    Returns:
        bool: - `True` if the number of tokens in the input text is less than or equal to `max_token_for_model`.
              - `False` otherwise.
    """
    enc = tiktoken.encoding_for_model(embedding_model)
    n_tokens = len(enc.encode(text=text))
    if n_tokens <= max_token_for_model:
        return True
    else:
        return False

In [21]:
from azure.core.exceptions import AzureError

from io import BytesIO
import pandas as pd
from tqdm import tqdm
from azure.core.exceptions import AzureError

def process_and_upload_chunks(df_measures_to_upload, embedding_model,chunk_size=2):
    columns_embeddings = [
        "Name",
        "Description",
        "BusinessRequirement",
        "InitialState",
        "TargetState",
        "Analysis",
        "Remarks",
        "Results",
        "LessonsLearned",
    ]

    container_name = 'translated-measures'

    # Initialize the Blob Service Client
    blob_service_client = BlobServiceClient(
    account_url="https://satutorial001.blob.core.windows.net/", credential=credential)
    container_client = blob_service_client.get_container_client(container=container_name)
    
    # Sort the DataFrame by MeasureId
    df_measures_to_upload = df_measures_to_upload.sort_values(by='MeasureId')
    
    chunk_to_upload = []
    chunk_counter = 0

    # Initialize tqdm progress bar for this chunk
    for _, row in tqdm(df_measures_to_upload.iterrows(), total=len(df_measures_to_upload), desc="Processing measures"):
        for column in columns_embeddings:
            # Calculate Embeddings
            if row[column] is not None:
                if not check_token_size_before_embedding(text=row[column], max_token_for_model=8191):
                    print(f"Skipping measure {row['MeasureId']} due to token size")
                    continue
                row[column + "_vector"] = calculate_embeddings(row[column], embedding_model)
        # Transform into JSON and add to chunk
        chunk_to_upload.append(row.to_dict())

        # Check if chunk size is reached
        if len(chunk_to_upload) >= chunk_size:
            upload_chunk(chunk_to_upload, container_client, chunk_counter)
            chunk_to_upload = []
            chunk_counter += 1

    # Upload any remaining data
    if chunk_to_upload:
        upload_chunk(chunk_to_upload, container_client, chunk_counter)
        
    concatenate_and_cleanup_measures(container_client)

def upload_chunk(chunk_to_upload, container_client, chunk_counter):
    try:
        # Save the chunk to a DataFrame
        df_chunk = pd.DataFrame(chunk_to_upload)

        # Convert the DataFrame to a Parquet file
        parquet_buffer = BytesIO()
        df_chunk.to_parquet(parquet_buffer, index=False)
        parquet_buffer.seek(0)

        # Upload the Parquet file to the blob storage
        parquet_blob_client = container_client.get_blob_client(f"measures/measures_chunk_{chunk_counter}.parquet")
        parquet_blob_client.upload_blob(parquet_buffer, overwrite=True)
        
        # Print the last MeasureId from the chunk
        last_measure_id = df_chunk['MeasureId'].iloc[-1]
        print(f"Successfully uploaded chunk {chunk_counter}, last MeasureId: {last_measure_id}")

    except AzureError as e:
        print(f"Failed to upload chunk {chunk_counter}: {e}")

def concatenate_and_cleanup_measures(container_client):
    try:
        # List all blobs in the 'measures' folder
        blob_list = container_client.list_blobs(name_starts_with="measures/")
        df_list = []

        for blob in blob_list:
            blob_client = container_client.get_blob_client(blob)
            blob_data = blob_client.download_blob().readall()
            df_chunk = pd.read_parquet(BytesIO(blob_data))
            df_list.append(df_chunk)

            # Delete the blob after reading
            blob_client.delete_blob()
            print(f"Deleted blob: {blob.name}")

        # Concatenate all DataFrames
        df_concatenated = pd.concat(df_list, ignore_index=True)

        # Convert the concatenated DataFrame to a Parquet file
        parquet_buffer = BytesIO()
        df_concatenated.to_parquet(parquet_buffer, index=False)
        parquet_buffer.seek(0)

        # Upload the Parquet file to the blob storage
        parquet_blob_client = container_client.get_blob_client("measures/measures_embeddings.parquet")
        parquet_blob_client.upload_blob(parquet_buffer, overwrite=True)
        print("Successfully created measures_embeddings.parquet")

    except AzureError as e:
        print(f"Failed to concatenate and cleanup measures: {e}")

        # Example usage
        # concatenate_and_cleanup_measures(container_client)

# Example usage
# process_and_upload_chunks(df_measures_to_upload, columns_embeddings, embedding_model, container_client, chunk_size=100)

In [25]:
index_name = "measures-data"
service_endpoint = "https://azsearch-tutorial-001.search.windows.net/"
azure_endpoint = "https://openai-tutorial-001.openai.azure.com/"
embedding_model = "text-embedding-ada-002"
api_version = "2023-05-15"
columns_relevant = [
        "measureId",
        "lastUpdatedAt",
        "deleted",
        "name",
        "description",
        "businessRequirement",
        "initialState",
        "targetState",
        "analysis",
        "remarks",
        "results",
        "lessonsLearned",
    ]
embedding_vector_size = retrieve_measures_from_AI_search_service()
df = read_translated_measures_data_for_initial_load()
df_measures_to_upload = reduce_raw_dataframe_to_relevant_columns(df, columns_relevant)
df_measures_with_initialization = initialize_embedding_vectors(df_measures_to_upload,columns_relevant,embedding_vector_size)
process_and_upload_chunks(df_measures_with_initialization,embedding_model)

----------------------------------------------------------------------------------------------------
Step 3: Retrieve data from AI Search Service to get state of measures...
No index found with the name 'measures-data'. Attempting to create the index...
Index measures-data created


C:\Users\DANE\AppData\Local\Temp\ipykernel_13732\2126082853.py:16: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_translations['createdAt'] = pd.to_datetime(
C:\Users\DANE\AppData\Local\Temp\ipykernel_13732\2126082853.py:18: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_translations['endDate'] = pd.to_datetime(
C:\Users\DANE\AppData\Local\Temp\ipykernel_13732\2126082853.py:20: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_translations['startDate'] = pd.to_datetime(


----------------------------------------------------------------------------------------------------
Step 2: Reduce measures to relevant columns...
Relevant measures:
   MeasureId       LastUpdatedAt  Deleted  \
0          1 2024-08-07 07:21:00        0   
1          2 2024-08-07 07:21:00        0   
2          3 2024-08-07 07:21:00        0   
3          4 2024-08-07 07:21:00        0   
4          5 2024-08-07 07:21:00        0   

                                    Name  \
0                                    TBD   
1                   BljP CIH outsourcing   
2  MSE-CC reduction of indirect capacity   
3                                    TBD   
4                                    TBD   

                                         Description  \
0                          Measure without an action   
1      CIH production outsource to external supplier   
2  2 expats to be replace by locals+ 2 iHC reduction   
3       LOG reduction on transport and custom duties   
4  BLC lower shar

Initialize vector embeddings: 100%|██████████| 12/12 [00:00<00:00, 1638.99it/s]


Size of data frame with measures to update: (6, 21)
  MeasureId       LastUpdatedAt  Deleted  \
0         1 2024-08-07 07:21:00        0   
1         2 2024-08-07 07:21:00        0   
2         3 2024-08-07 07:21:00        0   
3         4 2024-08-07 07:21:00        0   
4         5 2024-08-07 07:21:00        0   

                                    Name  \
0                                    TBD   
1                   BljP CIH outsourcing   
2  MSE-CC reduction of indirect capacity   
3                                    TBD   
4                                    TBD   

                                         Name_vector  \
0  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   
1  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   
2  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   
3  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   
4  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   

                                         Description  \
0                        

...

Successfully uploaded chunk 1, last MeasureId: 4


Processing measures: 100%|██████████| 6/6 [00:11<00:00,  1.91s/it]

Successfully uploaded chunk 2, last MeasureId: 6


Deleted blob: measures/measures_chunk_0.parquet
Deleted blob: measures/measures_chunk_1.parquet
Deleted blob: measures/measures_chunk_2.parquet


ArrowInvalid: Could not open Parquet input source '<Buffer>': Parquet magic bytes not found in footer. Either the file is corrupted or this is not a parquet file.

In [ ]:
    #columns_relevant = [
    #     "measureId",
    #     "lastUpdatedAt",
    #     "name",
    #     "description",
    #     "businessRequirement",
    #     "initialState",
    #     "targetState",
    #     "analysis",
    #     "remarks",
    #     "results",
    #     "lessonsLearned",
    # ]
# Calculate embeddings and save in blob storage
# 1. Read data from translated-measures blob storage
# 2. Generate embeddings for 
